<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:22px 28px; border-radius:8px">
  <div style="font-size:0.9em; letter-spacing:2px; opacity:0.85">NOTEBOOK 02 · FEATURE ENGINEERING · COMPLETED</div>
  <div style="font-size:2.0em; font-weight:700; margin-top:4px">🥈 Silver layer: a Lakeflow pipeline in SQL</div>
  <div style="font-size:1.1em; margin-top:8px; max-width:880px; opacity:0.95">
    Reshape the raw OMOP tables into three analytics-ready, per-patient feature views,
    declared entirely in SQL.
  </div>
</div>

## Why a *declarative* pipeline, and why SQL?

**Lakeflow Declarative Pipelines** let you describe *what* each table should contain; Databricks
works out the dependency graph, the incremental refresh, and the data-quality enforcement. We
write it in **SQL** on purpose. It's the simplest thing a clinical-data team can read, review,
and own.

Three silver feature views, one per concept the trial criteria care about:

| View | Grain | Feeds |
|---|---|---|
| `silver_biomarker_profile` | one row / patient | HER2 / ER / PR pivoted from `measurement` |
| `silver_prior_therapy` | one row / patient | anti-HER2 & endocrine therapy flags from `drug_exposure` |
| `silver_demographics` | one row / patient | age-at-diagnosis, menopausal status, AJCC stage |

<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px">
<b>The biomarker pivot is filled in below</b> (both in the pipeline source file and in the
run-it-now path), alongside the two worked-example views. The pivot is the classic long → wide
<code>MAX(CASE WHEN ...)</code> pattern.
</div>

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## ▸ Option A: the Lakeflow Declarative Pipeline (workshop path)

In a pipeline, `${source_catalog}` / `${source_schema}` are supplied as pipeline **configuration**
parameters, and `CONSTRAINT … EXPECT` clauses become enforced **data-quality expectations**.

### Creating & running the pipeline (the reproducible click-path)

1. Run the next cell to write `silver_layer.sql` to your workspace.
2. **Pipelines → Create pipeline → ETL pipeline.**
3. Name: `clinops_silver_biomarker_pipeline`. Serverless ✔, Photon ✔.
4. **Source code:** add the `silver_layer.sql` file.
5. **Destination:** Default catalog = your `catalog` widget value; Default schema = your `schema`.
6. **Configuration** (Advanced → Add): `source_catalog` = your catalog, `source_schema` = your schema.
7. **Create**, then **Run**. Watch the three views build in the pipeline graph.

In [0]:
import os

# All three materialized views, including the completed silver_biomarker_profile pivot.
SILVER_LAYER_SQL = """
-- ── silver_biomarker_profile (COMPLETED: long → wide pivot) ────────────────
-- One row per patient with her2_status / er_status / pr_status, pivoted from the
-- long measurement table. MAX(CASE WHEN ...) collapses the per-marker rows into one.
CREATE OR REFRESH MATERIALIZED VIEW silver_biomarker_profile (
  CONSTRAINT person_id_not_null EXPECT (person_id IS NOT NULL)
) COMMENT "Per-person structured biomarker pivot: HER2, ER, PR from the measurement table"
TBLPROPERTIES ("quality" = "silver") AS
WITH biomarkers AS (
  SELECT person_id,
    MAX(CASE WHEN measurement_source_value = 'HER2/neu'              THEN value_source_value END) AS her2_status,
    MAX(CASE WHEN measurement_source_value = 'Estrogen receptor'     THEN value_source_value END) AS er_status,
    MAX(CASE WHEN measurement_source_value = 'Progesterone receptor' THEN value_source_value END) AS pr_status
  FROM ${source_catalog}.${source_schema}.measurement
  WHERE measurement_source_value IN ('HER2/neu','Estrogen receptor','Progesterone receptor')
  GROUP BY person_id
)
SELECT person_id, her2_status, er_status, pr_status FROM biomarkers;

-- ── silver_prior_therapy (PRE-BUILT worked example) ───────────────────────
CREATE OR REFRESH MATERIALIZED VIEW silver_prior_therapy (
  CONSTRAINT person_id_not_null EXPECT (person_id IS NOT NULL)
) COMMENT "Per-person prior-therapy flags derived from drug_exposure"
TBLPROPERTIES ("quality" = "silver") AS
SELECT person_id,
  MAX(CASE WHEN drug_source_value IN ('Trastuzumab','Pertuzumab') THEN 1 ELSE 0 END) = 1 AS had_anti_her2_therapy,
  MAX(CASE WHEN drug_source_value IN ('Letrozole','Tamoxifen')   THEN 1 ELSE 0 END) = 1 AS had_endocrine_therapy
FROM ${source_catalog}.${source_schema}.drug_exposure
GROUP BY person_id;

-- ── silver_demographics (PRE-BUILT worked example) ────────────────────────
CREATE OR REFRESH MATERIALIZED VIEW silver_demographics (
  CONSTRAINT person_id_not_null EXPECT (person_id IS NOT NULL),
  CONSTRAINT age_plausible      EXPECT (age_at_dx_years BETWEEN 18 AND 110)
) COMMENT "Per-person demographics + diagnosis context for trial eligibility"
TBLPROPERTIES ("quality" = "silver") AS
WITH dx AS (
  SELECT person_id, MIN(condition_start_date) AS dx_date
  FROM ${source_catalog}.${source_schema}.condition_occurrence
  WHERE condition_source_value = 'Malignant neoplasm of breast'
  GROUP BY person_id
),
meno AS (
  SELECT person_id, MAX(value_source_value) AS menopausal_status
  FROM ${source_catalog}.${source_schema}.observation
  WHERE observation_source_value = 'Menopausal status'
  GROUP BY person_id
),
stage AS (
  SELECT person_id, MAX(value_source_value) AS ajcc_stage
  FROM ${source_catalog}.${source_schema}.observation
  WHERE observation_source_value = 'AJCC stage'
  GROUP BY person_id
)
SELECT p.person_id, p.gender_source_value,
  (year(dx.dx_date) - p.year_of_birth) AS age_at_dx_years,
  dx.dx_date, meno.menopausal_status, stage.ajcc_stage
FROM ${source_catalog}.${source_schema}.person p
JOIN dx         ON p.person_id = dx.person_id
LEFT JOIN meno  ON p.person_id = meno.person_id
LEFT JOIN stage ON p.person_id = stage.person_id;
""".strip()

out_dir = f"/Workspace/Users/{spark.sql('SELECT current_user()').first()[0]}/clinops_pipeline"
os.makedirs(out_dir, exist_ok=True)
with open(f"{out_dir}/silver_layer.sql", "w") as f:
    f.write(SILVER_LAYER_SQL)
print(f"✅ Pipeline source written to {out_dir}/silver_layer.sql")

✅ Pipeline source written to /Workspace/Users/ml-team@example.com/clinops_pipeline/silver_layer.sql


## ▸ Option B: run the same logic now (no pipeline required)

Identical SELECTs, materialized as tables right here so downstream notebooks have their inputs
immediately.

In [0]:
# Reads the read-only OMOP source via src(); writes to YOUR schema via fqn(). Using Python
# spark.sql keeps both fully qualified, so the read-only source and your writable schema can
# live in different schemas (or catalogs) with no session juggling.
spark.sql(f"""
CREATE OR REPLACE TABLE {fqn('silver_biomarker_profile')}
COMMENT 'Per-person structured biomarker pivot: HER2, ER, PR from measurement'
AS
WITH biomarkers AS (
  SELECT person_id,
    MAX(CASE WHEN measurement_source_value = 'HER2/neu'              THEN value_source_value END) AS her2_status,
    MAX(CASE WHEN measurement_source_value = 'Estrogen receptor'     THEN value_source_value END) AS er_status,
    MAX(CASE WHEN measurement_source_value = 'Progesterone receptor' THEN value_source_value END) AS pr_status
  FROM {src('measurement')}
  WHERE measurement_source_value IN ('HER2/neu','Estrogen receptor','Progesterone receptor')
  GROUP BY person_id
)
SELECT person_id, her2_status, er_status, pr_status FROM biomarkers
""")
print("✅ built", fqn('silver_biomarker_profile'))
display(spark.table(fqn('silver_biomarker_profile')).limit(10))

✅ built clinops_catalog.clinops_ml.silver_biomarker_profile


person_id,her2_status,er_status,pr_status
28,Positive,Positive,Negative
29,Positive,Negative,Positive
30,Positive,Positive,Positive
33,Negative,Positive,Negative
93,Negative,Positive,Positive
151,Negative,Positive,Negative
171,Positive,Positive,Positive
248,Negative,Positive,Positive
262,Positive,Positive,Positive
269,Negative,Negative,Negative


In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {fqn('silver_prior_therapy')}
COMMENT 'Per-person prior-therapy flags derived from drug_exposure'
AS
SELECT person_id,
  MAX(CASE WHEN drug_source_value IN ('Trastuzumab','Pertuzumab') THEN 1 ELSE 0 END) = 1 AS had_anti_her2_therapy,
  MAX(CASE WHEN drug_source_value IN ('Letrozole','Tamoxifen')   THEN 1 ELSE 0 END) = 1 AS had_endocrine_therapy
FROM {src('drug_exposure')}
GROUP BY person_id
""")
print("✅ built", fqn('silver_prior_therapy'))
display(spark.table(fqn('silver_prior_therapy')).limit(10))

✅ built clinops_catalog.clinops_ml.silver_prior_therapy


person_id,had_anti_her2_therapy,had_endocrine_therapy
28,true,false
29,true,false
30,true,false
93,false,true
171,false,true
200,false,false
221,false,false
232,false,true
295,false,true
9,false,true


In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {fqn('silver_demographics')}
COMMENT 'Per-person demographics + diagnosis context for trial eligibility'
AS
WITH dx AS (
  SELECT person_id, MIN(condition_start_date) AS dx_date
  FROM {src('condition_occurrence')}
  WHERE condition_source_value = 'Malignant neoplasm of breast'
  GROUP BY person_id
),
meno AS (
  SELECT person_id, MAX(value_source_value) AS menopausal_status
  FROM {src('observation')} WHERE observation_source_value = 'Menopausal status' GROUP BY person_id
),
stage AS (
  SELECT person_id, MAX(value_source_value) AS ajcc_stage
  FROM {src('observation')} WHERE observation_source_value = 'AJCC stage' GROUP BY person_id
)
SELECT p.person_id, p.gender_source_value,
  (year(dx.dx_date) - p.year_of_birth) AS age_at_dx_years,
  dx.dx_date, meno.menopausal_status, stage.ajcc_stage
FROM {src('person')} p
JOIN dx         ON p.person_id = dx.person_id
LEFT JOIN meno  ON p.person_id = meno.person_id
LEFT JOIN stage ON p.person_id = stage.person_id
""")
print("✅ built", fqn('silver_demographics'))
display(spark.table(fqn('silver_demographics')).limit(10))

✅ built clinops_catalog.clinops_ml.silver_demographics


person_id,gender_source_value,age_at_dx_years,dx_date,menopausal_status,ajcc_stage
1,FEMALE,35,2023-01-02,null,Stage IIA
2,FEMALE,38,2024-05-01,null,Stage IIB
3,FEMALE,58,2024-11-25,Postmenopausal,Stage IIIA
4,FEMALE,29,2024-04-27,null,Stage IIB
5,FEMALE,55,2024-03-29,Postmenopausal,Stage IIB
6,FEMALE,61,2023-07-19,Postmenopausal,Stage IIA
7,FEMALE,52,2023-05-30,Postmenopausal,Stage IIB
8,FEMALE,50,2023-08-21,Postmenopausal,Stage IIB
9,FEMALE,42,2023-06-01,null,Stage IIA
10,FEMALE,58,2023-10-28,Postmenopausal,Stage I


In [0]:
%sql
SELECT 'silver_biomarker_profile' AS view, COUNT(*) AS rows,
       SUM(CASE WHEN her2_status IS NOT NULL THEN 1 ELSE 0 END) AS has_her2
FROM silver_biomarker_profile
UNION ALL SELECT 'silver_prior_therapy',  COUNT(*), SUM(CASE WHEN had_anti_her2_therapy THEN 1 ELSE 0 END) FROM silver_prior_therapy
UNION ALL SELECT 'silver_demographics',   COUNT(*), SUM(CASE WHEN menopausal_status='Postmenopausal' THEN 1 ELSE 0 END) FROM silver_demographics;

view,rows,has_her2
silver_biomarker_profile,240,240
silver_prior_therapy,146,10
silver_demographics,300,206


<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px">
<b>What this builds:</b> three clean per-patient silver views. Note that
<code>silver_biomarker_profile</code> only has biomarkers for the ~240 patients with structured
<code>measurement</code> rows. The 60 <b>notes-only</b> patients are still missing. That gap is
exactly what notebook 03 quantifies and notebook 04 fills with NLP.
</div>

## ▶️ Next step
### → Open **[03_exploratory_data_analysis]($./03_exploratory_data_analysis)** to explore the data and see the gap.